# MTL (Multi-Task Learning) — External Evaluation Performance

Evaluasi model MTL biner (hERG, Cav1.2, Nav1.5) pada 3 level external set: easy, medium, hard.
Setiap model (7 konfigurasi fitur) diuji sekaligus untuk ketiga endpoint menggunakan satu forward pass MTLModularNet.

## SECTION 1: Imports & Global Config

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import ast
import pickle
import csv
from functools import partial

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

import matplotlib.pyplot as plt

# ================== GLOBAL CONFIG ==================

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS   = 0
PIN_MEMORY    = True
NON_BLOCKING  = True
KEY_COL       = "SMILES"
LABEL_COL     = "Activity"
ALL_ENDPOINTS = ["hERG", "Cav1.2", "Nav1.5"]

# FIX #1 — META_MD_COLS dilengkapi
META_MD_COLS = [
    # Identitas molekul
    "SMILES", "InChIKey", "Compound_ID", "CompoundID",
    # Label & aktivitas
    "Activity", "Activity_Label", "ActivityLabel", "pIC50",
    # Sumber
    "Source", "Source.1",
    # Tanimoto similarity
    "Max_Tanimoto_to_Dev", "Max_Tanimoto_to_Dev.1",
    "MaxTanimototoDev", "Max_Tanimoto",
]

print(f"[INFO] DEVICE = {DEVICE}")
print(f"[INFO] META_MD_COLS ({len(META_MD_COLS)} cols) defined.")

## SECTION 2: Paths & Hyperparameters

In [ ]:
# ====================== PATHS & HYPERPARAMS ======================
MODEL_PATHS = {
    "FP":          r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP\final\final_mtl_model.pt",
    "MD":          r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD\final\final_mtl_model.pt",
    "Graph":       r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\Graph\final\final_mtl_model.pt",
    "FP+MD":       r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD\final\final_mtl_model.pt",
    "FP+Graph":    r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+Graph\final\final_mtl_model.pt",
    "MD+Graph":    r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD+Graph\final\final_mtl_model.pt",
    "FP+MD+Graph": r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD+Graph\final\final_mtl_model.pt",
}

SCALER_PATHS = {
    "MD":          r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD\final\scaler_mordred.pkl",
    "FP+MD":       r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD\final\scaler_mordred.pkl",
    "MD+Graph":    r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD+Graph\final\scaler_mordred.pkl",
    "FP+MD+Graph": r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD+Graph\final\scaler_mordred.pkl",
}

# FIX #2 — Path ke file kolom MD hasil intersection dari training
# File ini dibuat otomatis oleh align_mordred_cols() saat training
# Isinya: daftar kolom MD yang sudah di-sort & di-align antar endpoint
SHARED_MD_COLS_PATH = r"D:\QSAR Cardiotoxicity\Binary_Multitask\md_aligned_feature_cols.txt"

# Hyperparameter per konfigurasi — hidden_dim dibaca dari state_dict langsung
# supaya tidak perlu hardcode dan tidak mismatch
CONFIG_HP = {
    "FP":          {"arch_type": "pyramid",    "n_layers": 2, "hidden_dim": 387,  "dropout": 0.2833336280599541},
    "MD":          {"arch_type": "wide_first", "n_layers": 4, "hidden_dim": 390,  "dropout": 0.08821},
    "Graph":       {"arch_type": "wide_first", "n_layers": 4, "hidden_dim": None, "dropout": 0.4685182959600666},  # hidden_dim=None → auto dari state_dict
    "FP+MD":       {"arch_type": "uniform",    "n_layers": 2, "hidden_dim": 335,  "dropout": 0.239746},
    "FP+Graph":    {"arch_type": "wide_first", "n_layers": 4, "hidden_dim": None, "dropout": 0.4685182959600666},  # hidden_dim=None → auto dari state_dict
    "MD+Graph":    {"arch_type": "uniform",    "n_layers": 4, "hidden_dim": 423,  "dropout": 0.243376098286401},
    "FP+MD+Graph": {"arch_type": "uniform",    "n_layers": 2, "hidden_dim": 338,  "dropout": 0.11237549477045344},
}

# ====================== EXTERNAL DATASETS ======================
external_sets = {
    "easy": {
        "hERG": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal1_easy_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\hERG\MD_Aligned_to_DevSet\hERG_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Cav1.2": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal1_easy_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Nav1.5": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal1_easy_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"
        }
    },
    "medium": {
        "hERG": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal2_medium_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\hERG\MD_Aligned_to_DevSet\hERG_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Cav1.2": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal2_medium_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Nav1.5": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal2_medium_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"
        }
    },
    "hard": {
        "hERG": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal3_hard_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\hERG\MD_Aligned_to_DevSet\hERG_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Cav1.2": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal3_hard_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Nav1.5": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal3_hard_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"
        }
    }
}

OUT_DIR = r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL_Evaluation"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"[INFO] Output dir: {OUT_DIR}")

In [ ]:
# DEBUG: cek ukuran semua file per level & endpoint
for level in ["easy", "medium", "hard"]:
    for endpoint in ALL_ENDPOINTS:
        paths = external_sets[level][endpoint]
        
        df_fp = pd.read_csv(paths["fp"])
        graphs_raw, _ = load_graphs_from_pkl(paths["graph"])
        df_md = pd.read_excel(paths["md"])
        
        n_fp    = len(df_fp)
        n_graph = len(graphs_raw)
        n_md    = len(df_md)
        
        ok = "✅" if n_fp == n_graph == n_md else "❌ MISMATCH"
        print(f"{ok}  {level:6s} | {endpoint:6s} | FP={n_fp}  Graph={n_graph}  MD={n_md}")

In [ ]:
# -*- coding: utf-8 -*-
"""
DIAGNOSTIC: Feature Dimension pada External Validation Sets
Binary Multitask MTL (hERG, Cav1.2, Nav1.5)
"""

import os
import json
import ast
import pickle
import numpy as np
import pandas as pd

# ====================== EXTERNAL SETS (copy dari kode Anda) ======================
external_sets = {
    "easy": {
        "hERG": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal1_easy_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\hERG\MD_Aligned_to_DevSet\hERG_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Cav1.2": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal1_easy_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Nav1.5": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal1_easy_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"
        }
    },
    "medium": {
        "hERG": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal2_medium_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\hERG\MD_Aligned_to_DevSet\hERG_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Cav1.2": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal2_medium_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Nav1.5": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal2_medium_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"
        }
    },
    "hard": {
        "hERG": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\hERG_ExternalVal3_hard_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\hERG\MD_Aligned_to_DevSet\hERG_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Cav1.2": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Cav1.2_ExternalVal3_hard_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"
        },
        "Nav1.5": {
            "fp":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
            "graph": r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Cav1.2\MD files khusus\Nav1.5_ExternalVal3_hard_graphs_67feat_with_labels.pkl",
            "md":    r"D:\QSAR Cardiotoxicity\Binary_Models\Performance Evaluation\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"
        }
    }
}

META_MD_COLS = [
    "SMILES", "Activity", "pIC50", "Source", "Activity_Label",
    "Compound_ID", "InChIKey", "MaxTanimototoDev", "Max_Tanimoto_to_Dev",
    "ActivityLabel", "CompoundID", "Max_Tanimoto_to_Dev.1"
]

# ====================== HELPER FUNCTIONS ======================
def fast_parse_list(s):
    if isinstance(s, (list, np.ndarray)):
        return s.tolist() if isinstance(s, np.ndarray) else s
    if not isinstance(s, str):
        return s
    try:
        return json.loads(s)
    except:
        return ast.literal_eval(s)

def get_fp_dimension(df_fp, ec_col="ECFP4", mc_col="MACCS"):
    ec = fast_parse_list(df_fp[ec_col].iloc[0])
    mc = fast_parse_list(df_fp[mc_col].iloc[0])
    return len(ec), len(mc), len(ec) + len(mc)

def get_md_dimension(df_md):
    feature_cols = [c for c in df_md.columns 
                    if c not in META_MD_COLS 
                    and pd.api.types.is_numeric_dtype(df_md[c])]
    return len(feature_cols), feature_cols[:6]  # tampilkan 6 contoh

def get_graph_node_dim(pkl_path):
    with open(pkl_path, "rb") as f:
        gobj = pickle.load(f)
    # ambil item pertama
    item = gobj[0]
    if isinstance(item, dict) and "graph" in item:
        gdict = item["graph"]
    else:
        gdict = item
    node_feat = np.asarray(gdict["node_features"], dtype=np.float32)
    return node_feat.shape[1]

# ====================== MAIN DIAGNOSTIC ======================
print("🔍 FEATURE DIMENSION DIAGNOSTIC — Binary Multitask MTL\n")
print("=" * 110)

for level in ["easy", "medium", "hard"]:
    print(f"\n📌 LEVEL: {level.upper()}")
    print("-" * 110)
    
    for endpoint in ["hERG", "Cav1.2", "Nav1.5"]:
        paths = external_sets[level][endpoint]
        print(f"   🔹 {endpoint:7} | Level: {level.upper():7}")
        
        # 1. Fingerprint (ECFP4 + MACCS)
        if os.path.exists(paths["fp"]):
            df_fp = pd.read_csv(paths["fp"])
            ec_dim, mc_dim, total_fp = get_fp_dimension(df_fp)
            print(f"      FP     : ECFP4 = {ec_dim:4d}  |  MACCS = {mc_dim:3d}  |  Total FP = {total_fp:4d}")
        else:
            print(f"      FP     : ❌ File tidak ditemukan")
        
        # 2. Mordred (MD)
        if os.path.exists(paths["md"]):
            df_md = pd.read_excel(paths["md"])
            md_dim, sample_cols = get_md_dimension(df_md)
            print(f"      MD     : {md_dim:4d} fitur  (contoh: {sample_cols})")
        else:
            print(f"      MD     : ❌ File tidak ditemukan")
        
        # 3. Graph
        if os.path.exists(paths["graph"]):
            g_dim = get_graph_node_dim(paths["graph"])
            print(f"      Graph  : Node features = {g_dim:3d}")
        else:
            print(f"      Graph  : ❌ File tidak ditemukan")
        
        print()

print("=" * 110)
print("✅ Diagnostic selesai. Semua dimensi fitur sudah terlihat di atas.")

In [ ]:
# -*- coding: utf-8 -*-
"""
INSPECT INPUT FEATURE DIMENSIONS — Binary Multitask MTL Models
[VERSI FIXED - Handle None safely]
"""

import os
import torch
import pickle
import pandas as pd

# ====================== PATHS ======================
MODEL_PATHS = {
    "FP":          r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP\final\final_mtl_model.pt",
    "MD":          r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD\final\final_mtl_model.pt",
    "Graph":       r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\Graph\final\final_mtl_model.pt",
    "FP+MD":       r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD\final\final_mtl_model.pt",
    "FP+Graph":    r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+Graph\final\final_mtl_model.pt",
    "MD+Graph":    r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD+Graph\final\final_mtl_model.pt",
    "FP+MD+Graph": r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD+Graph\final\final_mtl_model.pt",
}

SCALER_PATHS = {
    "MD":          r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD\final\scaler_mordred.pkl",
    "FP+MD":       r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD\final\scaler_mordred.pkl",
    "MD+Graph":    r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\MD+Graph\final\scaler_mordred.pkl",
    "FP+MD+Graph": r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL\FP+MD+Graph\final\scaler_mordred.pkl",
}

print("🔍 INSPECTING INPUT FEATURE DIMENSIONS FROM MODEL STATE_DICT\n")
print("=" * 100)

results = []

for config_name, model_path in MODEL_PATHS.items():
    if not os.path.exists(model_path):
        print(f"❌ {config_name:12} → File tidak ditemukan")
        continue

    state = torch.load(model_path, map_location='cpu', weights_only=True)

    # === FP Encoder ===
    fp_dim = None
    if 'fp_encoder.net.0.weight' in state:
        fp_dim = state['fp_encoder.net.0.weight'].shape[1]
    else:
        for k in state.keys():
            if 'fp_encoder' in k and '.0.weight' in k:
                fp_dim = state[k].shape[1]
                break

    # === MD Encoder ===
    md_dim = None
    if 'md_encoder.net.0.weight' in state:
        md_dim = state['md_encoder.net.0.weight'].shape[1]
    else:
        for k in state.keys():
            if 'md_encoder' in k and '.0.weight' in k:
                md_dim = state[k].shape[1]
                break

    # === Graph Encoder ===
    g_dim = None
    if 'g_encoder.conv1.lin.weight' in state:           # PyG 2.x
        g_dim = state['g_encoder.conv1.lin.weight'].shape[1]
    elif 'g_encoder.conv1.weight' in state:             # PyG lama
        g_dim = state['g_encoder.conv1.weight'].shape[1]
    else:
        for k in state.keys():
            if 'g_encoder.conv1' in k and 'weight' in k:
                g_dim = state[k].shape[1]
                break

    # Hitung jumlah task (konfirmasi multitask)
    task_backbones = sum(1 for k in state.keys() if k.startswith('task_backbones.'))

    results.append({
        "Config":          config_name,
        "FP_input_dim":    fp_dim,
        "MD_input_dim":    md_dim,
        "Graph_node_dim":  g_dim,
        "Tasks_detected":  task_backbones,
        "File_size_MB":    round(os.path.getsize(model_path) / (1024*1024), 2)
    })

    # Safe printing (handle None)
    fp_str = f"{fp_dim:5}" if fp_dim is not None else "  None"
    md_str = f"{md_dim:5}" if md_dim is not None else "  None"
    g_str  = f"{g_dim:5}"  if g_dim  is not None else "  None"

    print(f"✅ {config_name:12} → FP: {fp_str} | MD: {md_str} | Graph: {g_str}")

print("\n" + "=" * 100)
print("🔍 MORDRED SCALER CHECK")
for name, path in SCALER_PATHS.items():
    if os.path.exists(path):
        with open(path, "rb") as f:
            scaler = pickle.load(f)
        n_feat = getattr(scaler, "n_features_in_", "Unknown")
        print(f"   {name:12} scaler → {n_feat:4} fitur")
    else:
        print(f"   {name:12} scaler → ❌ File tidak ditemukan")

# ====================== TABEL RINGKASAN ======================
df = pd.DataFrame(results)
print("\n" + "=" * 100)
print("📋 RINGKASAN INPUT DIMENSIONS")
print("=" * 100)
print(df.to_string(index=False))

# Simpan ke Excel
output_excel = r"D:\QSAR Cardiotoxicity\Binary_Multitask\MTL_Evaluation\Model_Input_Dimensions.xlsx"
df.to_excel(output_excel, index=False)
print(f"\n💾 Tabel berhasil disimpan ke:\n   {output_excel}")

## SECTION 3: Helper — Parsing FP & MD

In [ ]:
# ================== HELPER: FP & MD PARSING ==================

def fast_parse_list(s):
    if isinstance(s, list):       return s
    if isinstance(s, np.ndarray): return s.tolist()
    if not isinstance(s, str):    raise TypeError(f"Unexpected type: {type(s)}")
    try:    return json.loads(s)
    except: return ast.literal_eval(s)


def compute_fp_matrix_from_df(df_fp, ec_col="ECFP4", mc_col="MACCS"):
    """Gabungkan ECFP4 + MACCS menjadi satu matrix numpy."""
    ec = df_fp[ec_col].tolist()
    mc = df_fp[mc_col].tolist()
    ec0 = fast_parse_list(ec[0])
    mc0 = fast_parse_list(mc[0])
    d_ec, d_mc = len(ec0), len(mc0)
    X = np.zeros((len(df_fp), d_ec + d_mc), dtype=np.float32)
    for i in range(len(df_fp)):
        e = np.asarray(fast_parse_list(ec[i]), dtype=np.float32)
        m = np.asarray(fast_parse_list(mc[i]), dtype=np.float32)
        X[i, :d_ec] = e
        X[i, d_ec:] = m
    return X, d_ec, d_mc


# FIX #3 — Baca SHARED_MD_COLS dari file txt yang disimpan saat training
# File ini berisi urutan kolom MD yang persis sama dengan saat scaler di-fit
def load_shared_md_cols(path=SHARED_MD_COLS_PATH):
    """
    Baca daftar kolom MD dari file txt yang disimpan training.
    Format file: baris pertama = komentar/header, sisanya = nama kolom (1 per baris).
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[ERROR] File SHARED_MD_COLS tidak ditemukan:\n  {path}\n"
            f"Pastikan path sudah benar dan training sudah dijalankan."
        )
    cols = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            # Skip baris header/komentar (Strategy:, N features:, baris kosong)
            if not line or line.startswith("Strategy") or line.startswith("N features"):
                continue
            cols.append(line)
    print(f"[INFO] SHARED_MD_COLS loaded: {len(cols)} kolom dari {path}")
    return cols


def get_mordred_feature_cols(df_md, shared_cols=None, meta_cols=META_MD_COLS):
    """
    Ambil kolom fitur MD dengan urutan yang dijamin konsisten dengan training.

    Prioritas:
      1. shared_cols (dari file txt training) → PALING AMAN, urutan dijamin sama
      2. Fallback: filter numerik non-meta dari DataFrame (dengan WARNING)
    """
    if shared_cols is not None:
        missing = [c for c in shared_cols if c not in df_md.columns]
        if missing:
            raise ValueError(
                f"[ERROR] Kolom dari shared_md_cols tidak ada di df_md:\n{missing}"
            )
        return shared_cols

    # Fallback — hanya jika file txt tidak tersedia
    print("    [WARN] shared_cols=None — pakai fallback filter (urutan tidak dijamin!)")
    return [
        c for c in df_md.columns
        if c not in meta_cols and pd.api.types.is_numeric_dtype(df_md[c])
    ]


def load_graphs_from_pkl(path):
    """Load file .pkl graph, konversi dict ke torch_geometric.Data jika perlu."""
    with open(path, "rb") as f:
        gobj = pickle.load(f)

    graphs = []
    labels = []
    for item in gobj:
        if isinstance(item, dict):
            gdict = item.get("graph", item)
            xnp   = np.asarray(gdict["node_features"], dtype=np.float32)
            einp  = np.asarray(gdict["edge_index"],    dtype=np.int64).T
            data  = Data(
                x          = torch.from_numpy(xnp),
                edge_index = torch.from_numpy(einp),
            )
            graphs.append(data)
            labels.append(int(item["Activity"]))
        elif isinstance(item, Data):
            graphs.append(item)
            labels.append(int(item.y))
        else:
            raise TypeError(f"Tipe graph tidak dikenali: {type(item)}")

    return graphs, np.array(labels, dtype=int)


# Load SHARED_MD_COLS sekali di sini — dipakai di Section 6
SHARED_MD_COLS = load_shared_md_cols(SHARED_MD_COLS_PATH)

print("[OK] Helper functions defined.")

## SECTION 4: Dataset, DataLoader, Model Architecture (MTL)

In [ ]:
# ================== MTL DATASET & DATALOADER ==================

class MTLQSARDataset(Dataset):
    """
    Dataset untuk evaluasi MTL.
    Y: np.ndarray shape (N, n_tasks) — kolom yang tidak relevan diisi -1 (masked).
    """
    def __init__(self, X_fp, X_md, graphs, Y):
        self.X_fp   = X_fp
        self.X_md   = X_md
        self.graphs = graphs
        self.Y      = Y
        n = len(Y)
        if X_fp   is not None: assert X_fp.shape[0] == n,  "X_fp size mismatch"
        if X_md   is not None: assert X_md.shape[0] == n,  "X_md size mismatch"
        if graphs is not None: assert len(graphs)   == n,  "graphs size mismatch"

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        x_fp = torch.from_numpy(self.X_fp[idx]).float() if self.X_fp   is not None else None
        x_md = torch.from_numpy(self.X_md[idx]).float() if self.X_md   is not None else None
        g    = self.graphs[idx]                          if self.graphs is not None else None
        y    = torch.tensor(self.Y[idx], dtype=torch.long)
        return x_fp, x_md, g, y


def collate_fn_mtl(batch):
    x_fp_list, x_md_list, g_list, y_list = zip(*batch)
    has_fp = any(x is not None for x in x_fp_list)
    has_md = any(x is not None for x in x_md_list)
    has_g  = any(g is not None for g in g_list)
    x_fp_batch = torch.stack(list(x_fp_list), 0) if has_fp else None
    x_md_batch = torch.stack(list(x_md_list), 0) if has_md else None
    g_batch    = Batch.from_data_list([g for g in g_list if g is not None]) if has_g else None
    y_batch    = torch.stack(list(y_list), 0)
    return x_fp_batch, x_md_batch, g_batch, y_batch


def make_mtl_loader(X_fp, X_md, graphs, Y, batch_size=64, shuffle=False):
    ds = MTLQSARDataset(X_fp, X_md, graphs, Y)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        collate_fn=collate_fn_mtl,
    )


# ================== MODEL ARCHITECTURE (MTL) ==================

class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)


class GraphEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=128, dropout=0.0):
        super().__init__()
        self.conv1   = GCNConv(in_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, batch):
        x, ei, bidx = batch.x, batch.edge_index, batch.batch
        h = F.relu(self.conv1(x, ei))
        h = self.dropout(h)
        h = F.relu(self.conv2(h, ei))
        return global_mean_pool(h, bidx)   # sama persis dengan training


def _build_backbone(concat_dim, hidden_dim, n_layers, arch_type, dropout):
    """
    Bangun shared backbone sesuai arch_type.
    Identik dengan fungsi yang dipakai saat training.
    """
    arch_type = (arch_type or "uniform").lower().replace("_", "")
    n_layers  = max(int(n_layers), 1)

    if arch_type == "uniform":
        layer_dims = [concat_dim] + [hidden_dim] * n_layers
    elif arch_type == "pyramid":
        dims = [concat_dim]; cur = hidden_dim
        for _ in range(n_layers):
            dims.append(cur); cur = max(cur // 2, 32)
        layer_dims = dims
    elif arch_type == "widefirst":
        layer_dims = [concat_dim, hidden_dim * 2] + [hidden_dim] * max(n_layers - 1, 0)
    else:
        layer_dims = [concat_dim] + [hidden_dim] * n_layers

    layers = []
    for in_d, out_d in zip(layer_dims[:-1], layer_dims[1:]):
        layers += [nn.Linear(in_d, out_d), nn.ReLU(), nn.Dropout(dropout)]
    return nn.Sequential(*layers), layer_dims[-1]


class MTLModularNet(nn.Module):
    """
    Shared backbone + task-specific heads (satu head per endpoint).
    Forward mengembalikan: dict {endpoint_name: logits (B, 2)}
    """
    def __init__(
        self,
        in_fp=0, in_md=0, in_g=None,
        use_fp=True, use_md=True, use_g=False,
        hidden_dim=128, dropout=0.2,
        n_layers=2, arch_type="uniform",
        task_names=None,
    ):
        super().__init__()
        self.task_names      = task_names or ALL_ENDPOINTS
        self.safe_task_names = [t.replace(".", "_") for t in self.task_names]
        self.name_map        = {o: s for o, s in zip(self.task_names, self.safe_task_names)}

        self.use_fp = use_fp and (in_fp > 0)
        self.use_md = use_md and (in_md > 0)
        self.use_g  = use_g  and (in_g is not None) and (in_g > 0)

        fp_out = md_out = g_out = 0
        if self.use_fp:
            self.fp_encoder = MLPEncoder(in_fp, hidden_dim, hidden_dim, dropout)
            fp_out = hidden_dim
        if self.use_md:
            self.md_encoder = MLPEncoder(in_md, hidden_dim, hidden_dim, dropout)
            md_out = hidden_dim
        if self.use_g:
            self.g_encoder  = GraphEncoder(in_g, 64, hidden_dim, dropout)
            g_out  = hidden_dim

        concat_dim = fp_out + md_out + g_out
        assert concat_dim > 0, "Minimal satu modalitas harus aktif."

        self.backbone, bb_out = _build_backbone(
            concat_dim, hidden_dim, n_layers, arch_type, dropout
        )
        self.heads = nn.ModuleDict({
            safe_t: nn.Linear(bb_out, 2) for safe_t in self.safe_task_names
        })

    def forward(self, x_fp=None, x_md=None, g_batch=None):
        feats = []
        if self.use_fp and x_fp    is not None: feats.append(self.fp_encoder(x_fp))
        if self.use_md and x_md    is not None: feats.append(self.md_encoder(x_md))
        if self.use_g  and g_batch is not None: feats.append(self.g_encoder(g_batch))
        assert len(feats) > 0, "Semua input None saat forward."
        h = feats[0] if len(feats) == 1 else torch.cat(feats, dim=1)
        z = self.backbone(h)
        return {orig: self.heads[self.name_map[orig]](z) for orig in self.task_names}


print("[OK] Dataset, DataLoader, dan Model MTL defined.")

## SECTION 5: Builder, Evaluate MTL, Bootstrap CI, Plot CM

In [ ]:
# ================== BUILDER MODEL MTL ==================

def build_mtl_model(config_name, in_fp=0, in_md=0, in_g=None, hidden_dim_override=None):
    """
    Bangun MTLModularNet sesuai nama konfigurasi.
    hidden_dim_override: jika tidak None, override CONFIG_HP[hidden_dim].
    Dipakai untuk Graph & FP+Graph yang hidden_dim-nya dibaca dari state_dict.
    """
    hp     = CONFIG_HP[config_name]
    use_fp = "FP"    in config_name
    use_md = "MD"    in config_name
    use_g  = "Graph" in config_name

    hidden_dim = hidden_dim_override if hidden_dim_override is not None else hp["hidden_dim"]

    return MTLModularNet(
        in_fp      = in_fp  if use_fp else 0,
        in_md      = in_md  if use_md else 0,
        in_g       = in_g   if use_g  else None,
        use_fp     = use_fp,
        use_md     = use_md,
        use_g      = use_g,
        hidden_dim = hidden_dim,
        dropout    = hp["dropout"],
        n_layers   = hp["n_layers"],
        arch_type  = hp["arch_type"],
        task_names = ALL_ENDPOINTS,
    ).to(DEVICE)


# ================== EVALUATE MTL (per endpoint) ==================

def evaluate_mtl_endpoint(model, loader, endpoint):
    """
    Jalankan inference MTL, ambil prediksi untuk `endpoint` saja.
    Hanya sample dengan label >= 0 yang dihitung (label -1 = masked).

    Returns: metrics (dict), all_y, all_prob, all_pred
    """
    ep_idx = ALL_ENDPOINTS.index(endpoint)
    model.eval()
    all_y, all_prob, all_pred = [], [], []

    with torch.no_grad():
        for x_fp, x_md, g_batch, y_batch in loader:
            y_batch = y_batch.to(DEVICE)
            if x_fp    is not None: x_fp    = x_fp.to(DEVICE,    non_blocking=NON_BLOCKING)
            if x_md    is not None: x_md    = x_md.to(DEVICE,    non_blocking=NON_BLOCKING)
            if g_batch is not None: g_batch = g_batch.to(DEVICE)

            logits_dict = model(x_fp=x_fp, x_md=x_md, g_batch=g_batch)
            logits_ep   = logits_dict[endpoint]          # (B, 2)

            probs = torch.softmax(logits_ep, dim=1)[:, 1].cpu().numpy()
            preds = (probs > 0.5).astype(int)

            y_ep = y_batch[:, ep_idx].cpu().numpy()
            mask = y_ep >= 0                             # abaikan label -1
            if mask.sum() == 0:
                continue

            all_y.append(y_ep[mask])
            all_prob.append(probs[mask])
            all_pred.append(preds[mask])

    all_y    = np.concatenate(all_y)
    all_prob = np.concatenate(all_prob)
    all_pred = np.concatenate(all_pred)

    try:
        auc = roc_auc_score(all_y, all_prob)
    except ValueError:
        auc = 0.0

    acc  = accuracy_score(all_y, all_pred)
    bacc = balanced_accuracy_score(all_y, all_pred)
    mcc  = matthews_corrcoef(all_y, all_pred)
    f1   = f1_score(all_y, all_pred, zero_division=0)

    tn, fp_c, fn, tp = confusion_matrix(all_y, all_pred, labels=[0, 1]).ravel()
    sen = tp / (tp + fn)   if (tp + fn)   > 0 else 0.0
    spe = tn / (tn + fp_c) if (tn + fp_c) > 0 else 0.0

    metrics = {
        "AUC": auc, "ACC": acc, "BACC": bacc,
        "SEN": sen, "SPE": spe, "MCC": mcc,  "F1": f1,
        "TP": tp,   "FP": fp_c, "TN": tn,    "FN": fn,
    }
    return metrics, all_y, all_prob, all_pred


# ================== BOOTSTRAP CI ==================

def bootstrap_ci_single(y_true, y_prob, y_pred, metric_fn,
                        n_boot=1000, alpha=0.95, random_state=42):
    """Bootstrap CI: returns (median, lower, upper, delta)."""
    rng    = np.random.RandomState(random_state)
    n      = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        y_b = y_true[idx]
        if len(np.unique(y_b)) < 2:
            continue
        try:
            s = metric_fn(y_b, y_prob[idx]) if metric_fn is roc_auc_score \
                else metric_fn(y_b, y_pred[idx])
            scores.append(s)
        except Exception:
            continue

    if not scores:
        return 0.0, 0.0, 0.0, 0.0

    scores = np.array(scores)
    lower  = np.percentile(scores, (1 - alpha) / 2 * 100)
    upper  = np.percentile(scores, (1 + alpha) / 2 * 100)
    median = np.median(scores)
    delta  = (upper - lower) / 2.0
    return median, lower, upper, delta


# ================== PLOT CONFUSION MATRIX ==================

def plot_confusion_matrix_cm(config_name, endpoint, level, cm, out_dir):
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)

    classes    = ["Neg", "Pos"]
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks); ax.set_xticklabels(classes)
    ax.set_yticks(tick_marks); ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"[MTL] {config_name} | {endpoint} | {level}")

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

    safe_cfg = config_name.replace("+", "_")
    fname    = os.path.join(out_dir, f"MTL_{safe_cfg}_{endpoint}_{level}_CM.png")
    plt.tight_layout()
    plt.savefig(fname, dpi=300)
    plt.close(fig)


print("[OK] Builder, evaluate, bootstrap, dan plot functions defined.")

## SECTION 6: CSV Setup & Main Evaluation Loop

In [ ]:
# ================== CSV SETUP ==================

csv_path = os.path.join(OUT_DIR, "MTL_external_performance.csv")

csv_header = [
    "Level", "Endpoint", "Config",
    "AUC",  "AUC_med",  "AUC_CI_lower",  "AUC_CI_upper",  "AUC_delta",  "AUC_display",
    "ACC",  "ACC_med",  "ACC_CI_lower",  "ACC_CI_upper",  "ACC_delta",  "ACC_display",
    "BACC", "BACC_med", "BACC_CI_lower", "BACC_CI_upper", "BACC_delta", "BACC_display",
    "MCC",  "MCC_med",  "MCC_CI_lower",  "MCC_CI_upper",  "MCC_delta",  "MCC_display",
    "F1",   "F1_med",   "F1_CI_lower",   "F1_CI_upper",   "F1_delta",   "F1_display",
    "SEN", "SPE", "TP", "FP", "TN", "FN",
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(csv_header)

print(f"[OK] CSV initialized: {csv_path}")

# FIX #4 — f1_safe untuk bootstrap agar zero_division konsisten
f1_safe = partial(f1_score, zero_division=0)


# ================== HELPER: baca hidden_dim dari state_dict ==================

def infer_hidden_dim_from_state(state):
    """
    Baca hidden_dim aktual dari state_dict checkpoint.
    Dipakai untuk config yang hidden_dim=None di CONFIG_HP (Graph, FP+Graph).
    Strategi: ambil output dim dari g_encoder.conv2.bias → itulah hidden_dim.
    """
    if "g_encoder.conv2.bias" in state:
        return state["g_encoder.conv2.bias"].shape[0]
    # Fallback: coba dari fp_encoder atau md_encoder
    for key in ["fp_encoder.net.0.weight", "md_encoder.net.0.weight"]:
        if key in state:
            return state[key].shape[0]  # output dim layer pertama = hidden_dim
    raise ValueError("[ERROR] Tidak bisa inferensi hidden_dim dari state_dict.")


# ================== MAIN EVALUATION LOOP ==================

CONFIGS_ALL = list(MODEL_PATHS.keys())

for level in ["easy", "medium", "hard"]:
    for config_name in CONFIGS_ALL:
        use_fp = "FP"    in config_name
        use_md = "MD"    in config_name
        use_g  = "Graph" in config_name

        # Load scaler sekali per config
        scaler_md = None
        if use_md and config_name in SCALER_PATHS:
            with open(SCALER_PATHS[config_name], "rb") as f:
                scaler_md = pickle.load(f)
            print(f"[INFO] scaler_md loaded for {config_name}")

        # ---- Tentukan dimensi dari endpoint pertama ----
        _ep0   = ALL_ENDPOINTS[0]
        _paths = external_sets[level][_ep0]
        _df_fp = pd.read_csv(_paths["fp"])
        _X_fp_tmp, _, _ = compute_fp_matrix_from_df(_df_fp, ec_col="ECFP4", mc_col="MACCS")
        _graphs_tmp, _  = load_graphs_from_pkl(_paths["graph"])

        in_fp = _X_fp_tmp.shape[1]        if use_fp else 0
        in_g  = _graphs_tmp[0].x.shape[1] if use_g  else None

        # FIX #3 — in_md dari SHARED_MD_COLS (urutan dijamin sama dengan training)
        in_md = 0
        if use_md:
            in_md = len(SHARED_MD_COLS)

        # ---- Baca hidden_dim dari state_dict jika None ----
        hp = CONFIG_HP[config_name]
        if hp["hidden_dim"] is None:
            _state_tmp = torch.load(MODEL_PATHS[config_name], map_location="cpu", weights_only=True)
            hidden_dim = infer_hidden_dim_from_state(_state_tmp)
            print(f"[INFO] {config_name}: hidden_dim auto-detected = {hidden_dim}")
            del _state_tmp
        else:
            hidden_dim = hp["hidden_dim"]

        # ---- Build & load model SEKALI per config ----
        print(f"\n--- Level: {level} | Config: {config_name} | "
              f"in_fp={in_fp}, in_md={in_md}, in_g={in_g}, hidden_dim={hidden_dim} ---")

        model = build_mtl_model(
            config_name, in_fp=in_fp, in_md=in_md, in_g=in_g,
            hidden_dim_override=hidden_dim
        )
        state = torch.load(MODEL_PATHS[config_name], map_location=DEVICE, weights_only=False)
        model.load_state_dict(state)
        model.eval()

        # ---- Loop endpoint ----
        for endpoint in ALL_ENDPOINTS:
            print(f"  >> {endpoint}", end=" ... ")
            paths = external_sets[level][endpoint]

            # Load FP
            df_fp = pd.read_csv(paths["fp"])
            X_fp_raw, _, _ = compute_fp_matrix_from_df(df_fp, ec_col="ECFP4", mc_col="MACCS")
            y_label = df_fp[LABEL_COL].astype(int).to_numpy()

            # Load Graph
            graphs_raw, y_g = load_graphs_from_pkl(paths["graph"])
            assert np.array_equal(y_label, y_g), \
                f"[ERROR] Label mismatch FP vs Graph | {level}/{endpoint}"

            # FIX #3 — Load & scale MD dengan urutan kolom dari SHARED_MD_COLS
            X_md_scaled = None
            if use_md:
                df_md        = pd.read_excel(paths["md"])
                md_feat_cols = get_mordred_feature_cols(df_md, shared_cols=SHARED_MD_COLS)
                X_md_raw     = (
                    df_md[md_feat_cols]
                    .fillna(0)
                    .replace([np.inf, -np.inf], 0)
                    .to_numpy(np.float32)
                )
                # Validasi shape sebelum transform
                assert X_md_raw.shape[1] == in_md, (
                    f"[ERROR] Shape MD mismatch! "
                    f"Expected {in_md}, got {X_md_raw.shape[1]} | "
                    f"{level}/{endpoint}/{config_name}"
                )
                X_md_scaled = scaler_md.transform(X_md_raw).astype(np.float32) \
                              if scaler_md is not None else X_md_raw

            X_fp   = X_fp_raw    if use_fp else None
            X_md   = X_md_scaled if use_md else None
            graphs = graphs_raw  if use_g  else None

            # Y_mtl
            N     = len(y_label)
            Y_mtl = np.full((N, len(ALL_ENDPOINTS)), fill_value=-1, dtype=np.int64)
            Y_mtl[:, ALL_ENDPOINTS.index(endpoint)] = y_label

            # DataLoader
            loader = make_mtl_loader(
                X_fp=X_fp, X_md=X_md, graphs=graphs, Y=Y_mtl,
                batch_size=64, shuffle=False,
            )

            # Evaluasi
            metrics, all_y, all_prob, all_pred = evaluate_mtl_endpoint(
                model, loader, endpoint
            )

            # FIX #4 — Bootstrap CI dengan f1_safe
            auc_med,  auc_lo,  auc_hi,  auc_d  = bootstrap_ci_single(all_y, all_prob, all_pred, roc_auc_score)
            acc_med,  acc_lo,  acc_hi,  acc_d  = bootstrap_ci_single(all_y, all_prob, all_pred, accuracy_score)
            bacc_med, bacc_lo, bacc_hi, bacc_d = bootstrap_ci_single(all_y, all_prob, all_pred, balanced_accuracy_score)
            mcc_med,  mcc_lo,  mcc_hi,  mcc_d  = bootstrap_ci_single(all_y, all_prob, all_pred, matthews_corrcoef)
            f1_med,   f1_lo,   f1_hi,   f1_d   = bootstrap_ci_single(all_y, all_prob, all_pred, f1_safe)

            auc_disp  = f"{auc_med:.4f} ± {auc_d:.4f}"
            acc_disp  = f"{acc_med:.4f} ± {acc_d:.4f}"
            bacc_disp = f"{bacc_med:.4f} ± {bacc_d:.4f}"
            mcc_disp  = f"{mcc_med:.4f} ± {mcc_d:.4f}"
            f1_disp   = f"{f1_med:.4f} ± {f1_d:.4f}"

            print(f"AUC={metrics['AUC']:.4f} | BACC={metrics['BACC']:.4f} "
                  f"| MCC={metrics['MCC']:.4f} | F1={metrics['F1']:.4f}")

            # Plot CM
            cm_val = confusion_matrix(all_y, all_pred, labels=[0, 1])
            plot_confusion_matrix_cm(config_name, endpoint, level, cm_val, OUT_DIR)

            # Tulis CSV
            with open(csv_path, "a", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow([
                    level, endpoint, config_name,
                    metrics["AUC"],
                    auc_med,  auc_lo,  auc_hi,  auc_d,  auc_disp,
                    metrics["ACC"],
                    acc_med,  acc_lo,  acc_hi,  acc_d,  acc_disp,
                    metrics["BACC"],
                    bacc_med, bacc_lo, bacc_hi, bacc_d, bacc_disp,
                    metrics["MCC"],
                    mcc_med,  mcc_lo,  mcc_hi,  mcc_d,  mcc_disp,
                    metrics["F1"],
                    f1_med,   f1_lo,   f1_hi,   f1_d,   f1_disp,
                    metrics["SEN"], metrics["SPE"],
                    metrics["TP"],  metrics["FP"],
                    metrics["TN"],  metrics["FN"],
                ])

print(f"\n[DONE] Hasil: {csv_path}")

## SECTION 7: Summary — Tampilkan Hasil CSV

In [ ]:
# ================== SUMMARY VIEW ==================

df_results = pd.read_csv(csv_path)

cols_show = [
    "Level", "Endpoint", "Config",
    "AUC", "BACC", "MCC", "F1", "ACC", "SEN", "SPE",
    "AUC_display", "BACC_display", "MCC_display", "F1_display",
]

print("=" * 110)
print("MTL EXTERNAL EVALUATION SUMMARY")
print("=" * 110)

for ep in ALL_ENDPOINTS:
    print(f"\n{'─'*90}")
    print(f"  Endpoint: {ep}")
    print(f"{'─'*90}")
    sub = df_results[df_results["Endpoint"] == ep][cols_show]
    print(sub.to_string(index=False))

print(f"\n[INFO] Total rows: {len(df_results)}")
print(f"[INFO] CSV path  : {csv_path}")

## SECTION 8: Visualisasi — Bar Chart AUC per Endpoint & Config

In [ ]:
# ================== VISUALISASI BAR CHART AUC ==================

df_results = pd.read_csv(csv_path)

configs = list(MODEL_PATHS.keys())
levels  = ["easy", "medium", "hard"]
colors  = {"easy": "#4CAF50", "medium": "#FF9800", "hard": "#F44336"}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for ax, endpoint in zip(axes, ALL_ENDPOINTS):
    df_ep = df_results[df_results["Endpoint"] == endpoint]
    x     = np.arange(len(configs))
    width = 0.25

    for i, level in enumerate(levels):
        df_lv  = df_ep[df_ep["Level"] == level].set_index("Config")
        aucs   = [df_lv.loc[c, "AUC"]       if c in df_lv.index else 0.0 for c in configs]
        deltas = [df_lv.loc[c, "AUC_delta"] if c in df_lv.index else 0.0 for c in configs]
        ax.bar(x + i * width, aucs, width, yerr=deltas,
               label=level, color=colors[level], capsize=4, alpha=0.85)

    ax.set_title(f"{endpoint}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Config")
    ax.set_ylabel("AUC")
    ax.set_xticks(x + width)
    ax.set_xticklabels(configs, rotation=30, ha="right", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.legend(title="Level", fontsize=9)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("MTL External Evaluation — AUC per Endpoint & Config (± 95% CI)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()

out_fig = os.path.join(OUT_DIR, "MTL_AUC_BarChart.png")
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.show()
print(f"[OK] Chart saved: {out_fig}")